In [ ]:
import faulthandler

faulthandler.enable()

In [ ]:
import sys
sys.path.append("../..")

import mytorch
import mytorch.nn as nn
import mytorch.nn.functional as F

DEVICE = "gpu:0"

In [ ]:
class FeedForwardModel(nn.Module):
    def __init__(self, device):
        super().__init__()
        self.l1 = nn.Linear(784, 196, device=device)
        self.relu1 = nn.ReLU()
        self.l2 = nn.Linear(196, 49, device=device)
        self.relu2 = nn.ReLU()
        self.l3 = nn.Linear(49, 24, device=device)
        self.relu3 = nn.ReLU()
        self.l4 = nn.Linear(24, 10, device=device)

    def forward(self, x):
        x = self.l1(x)
        x = self.relu1(x)
        x = self.l2(x)
        x = self.relu2(x)
        x = self.l3(x)
        x = self.relu3(x)
        x = self.l4(x)
        return x

In [ ]:
class CnnModel(nn.Module):
    def __init__(self, device):
        super().__init__()
        self.features = nn.Sequential([
            nn.Conv2d(1, 32, 3),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(32, 64, 3),
            nn.ReLU(),
            nn.MaxPool2d(2)
        ])
        self.classifier = nn.Sequential([
            nn.Linear(64 * 5 * 5, 128),
            nn.ReLU(),
            nn.Linear(128, 10)
        ])

    def forward(self, x):
        x = self.features(x)
        x = x.flatten(1)
        x = self.classifier(x)
        return x

In [ ]:
model = CnnModel(DEVICE)
sgd = mytorch.SGD(model.parameters(), 0.001, 0)

## Training

In [ ]:
import tqdm
from mnist_dataset import MnistDatset
from mytorch import AsyncDataLoader, MultiStepLR

NUM_EPOCHS = 10
BATCH_SIZE = 500

train_dataset = MnistDatset("dataset", True)
train_loader = AsyncDataLoader(train_dataset, BATCH_SIZE, device=DEVICE)
scheduler = MultiStepLR(sgd, [6, 8], 0.1)

In [ ]:
for epoch in range(NUM_EPOCHS):
    pbar = tqdm.tqdm(train_loader, desc=f"Epoch {epoch}")

    for X, y in pbar:
        logits = model.forward(X.reshape(-1, 1, 28, 28))
        loss = F.cross_entropy(logits, F.onehot(y, 10))
        pbar.set_postfix(loss="%.5f" % loss.item())

        sgd.zero_grad()
        loss.backward()
        sgd.step()
    
    scheduler.step()

## Inference / Test

In [ ]:
test_dataset = MnistDatset("dataset", False)
test_loader = AsyncDataLoader(test_dataset, BATCH_SIZE, device=DEVICE)

In [ ]:
import numpy as np

num_correct = 0
for X, y in tqdm.tqdm(test_loader):
    with mytorch.no_grad():
        logits = model.forward(X.reshape(-1, 1, 28, 28))
        prediction = logits.argmax(dim=1)
        prediction = prediction == y
        num_correct += np.sum(prediction.data)

print(100 * num_correct / len(test_dataset))